In [ ]:
import os
from pathlib import Path

# Set COCO_ROOT in your environment to override this local COCO dataset folder.
COCO_ROOT = Path(os.environ.get("COCO_ROOT", r"E:\projet\archive")).expanduser()
OUTPUT_DIR = Path("outputs/phase1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_image_dir(root, split):
    candidates = (root / split, root / split / split)
    for candidate in candidates:
        if candidate.is_dir() and any(candidate.glob("*.jpg")):
            return candidate

    matches = [
        path for path in root.rglob(split)
        if path.is_dir() and any(path.glob("*.jpg"))
    ] if root.is_dir() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise FileNotFoundError(f"Found multiple {split} folders under {root}.")
    return candidates[0]

TRAIN_IMG_DIR = find_image_dir(COCO_ROOT, "train2017")
VAL_IMG_DIR = find_image_dir(COCO_ROOT, "val2017")
ANNOTATIONS_DIR = COCO_ROOT / "annotations"
if not ANNOTATIONS_DIR.is_dir():
    ANNOTATIONS_DIR = COCO_ROOT / "annotations_trainval2017"

print(f"COCO data: {COCO_ROOT.resolve()}")
print(f"Training images: {TRAIN_IMG_DIR}")
print(f"Validation images: {VAL_IMG_DIR}")
print(f"Outputs: {OUTPUT_DIR.resolve()}")

In [ ]:
import csv
import hashlib
import json
import time
from collections import Counter, defaultdict
from pathlib import Path

import faiss
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from sentence_transformers import SentenceTransformer
from ultralytics import YOLO

TARGET_KB = 10_000
TARGET_VAL = 100
TARGET_TEST_1000 = 1_000
TARGET_TEST_100 = 100
YOLO_SAMPLE_SIZE = TARGET_KB
DETECTION_CONFIDENCE = 0.25
CAPTION_REFERENCE_CONFIDENCE = 1.0

sns.set_theme(style="whitegrid")

In [ ]:
def get_file_hash(path):
    hasher = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def scan_unique_images(candidates, seen_hashes=None):
    seen_hashes = set() if seen_hashes is None else seen_hashes
    clean_images = []
    audit = {
        "candidate_images": len(candidates),
        "examined_images": 0,
        "corrupt_images": 0,
        "duplicate_images": 0,
    }

    for img_path in candidates:
        audit["examined_images"] += 1
        try:
            with Image.open(img_path) as image:
                image.verify()
            image_hash = get_file_hash(img_path)
        except (OSError, ValueError, Image.DecompressionBombError):
            audit["corrupt_images"] += 1
            continue

        if image_hash in seen_hashes:
            audit["duplicate_images"] += 1
            continue
        seen_hashes.add(image_hash)
        clean_images.append(img_path)

    return clean_images, seen_hashes, audit


if not TRAIN_IMG_DIR.is_dir():
    raise FileNotFoundError(f"Training image folder not found: {TRAIN_IMG_DIR}")
if not VAL_IMG_DIR.is_dir():
    raise FileNotFoundError(f"Validation image folder not found: {VAL_IMG_DIR}")

print("Auditing every image and freezing the COCO splits...")
rng = np.random.default_rng(42)
train_candidates = sorted(TRAIN_IMG_DIR.glob("*.jpg"))
rng.shuffle(train_candidates)
clean_train_images, train_hashes, train_audit = scan_unique_images(train_candidates)

required_train_count = TARGET_KB + TARGET_VAL
if len(clean_train_images) < required_train_count:
    raise ValueError(
        f"Only {len(clean_train_images)} valid unique train images found; "
        f"{required_train_count} are required."
    )

kb_images = clean_train_images[:TARGET_KB]
val_images = clean_train_images[TARGET_KB:required_train_count]
assert not ({path.name for path in kb_images} & {path.name for path in val_images})

test_candidates = sorted(VAL_IMG_DIR.glob("*.jpg"))
rng.shuffle(test_candidates)
clean_test_images, _, test_audit = scan_unique_images(test_candidates, train_hashes)
if len(clean_test_images) < TARGET_TEST_1000:
    raise ValueError(
        f"Only {len(clean_test_images)} valid unique val images found; "
        f"{TARGET_TEST_1000} are required."
    )

test_1000 = clean_test_images[:TARGET_TEST_1000]
test_100 = test_1000[:TARGET_TEST_100]
splits = {
    "knowledge_base": [path.name for path in kb_images],
    "validation": [path.name for path in val_images],
    "test_1000": [path.name for path in test_1000],
    "test_100_compare": [path.name for path in test_100],
}

with (OUTPUT_DIR / "frozen_splits.json").open("w", encoding="utf-8") as file:
    json.dump(splits, file, indent=2)

cleaning_audit = {
    "duplicate_detection": "Exact SHA-256 file-content match",
    "train2017": {
        **train_audit,
        "valid_unique_images": len(clean_train_images),
        "selected_images": required_train_count,
        "valid_images_not_selected": len(clean_train_images) - required_train_count,
    },
    "val2017": {
        **test_audit,
        "valid_unique_images": len(clean_test_images),
        "selected_images": len(test_1000),
        "valid_images_not_selected": len(clean_test_images) - len(test_1000),
    },
    "split_sizes": {key: len(value) for key, value in splits.items()},
    "test_100_is_subset_of_test_1000": set(splits["test_100_compare"]).issubset(
        splits["test_1000"]
    ),
}
with (OUTPUT_DIR / "cleaning_audit.json").open("w", encoding="utf-8") as file:
    json.dump(cleaning_audit, file, indent=2)

print(f"Train2017: audited {train_audit['examined_images']:,}; selected {required_train_count:,} unique valid images")
print(f"  Knowledge base: {len(kb_images):,}; separate validation: {len(val_images):,}")
print(f"Val2017: audited {test_audit['examined_images']:,}; selected test: {len(test_1000):,}")
print(f"Comparison test: {len(test_100):,} (subset of the 1,000-image test)")
print(f"Cleaning audit saved to {OUTPUT_DIR / 'cleaning_audit.json'}")

In [ ]:
import csv
from collections import defaultdict

print("Exploring COCO ground truth and linking references to image IDs...")
instances_train_path = next(COCO_ROOT.rglob("instances_train2017.json"), None)
captions_train_path = next(COCO_ROOT.rglob("captions_train2017.json"), None)
instances_val_path = next(COCO_ROOT.rglob("instances_val2017.json"), None)
captions_val_path = next(COCO_ROOT.rglob("captions_val2017.json"), None)
required_annotation_files = {
    "instances_train2017.json": instances_train_path,
    "captions_train2017.json": captions_train_path,
    "instances_val2017.json": instances_val_path,
    "captions_val2017.json": captions_val_path,
}
missing_annotation_files = [name for name, path in required_annotation_files.items() if path is None]
if missing_annotation_files:
    raise FileNotFoundError(
        f"Missing COCO annotation files under {COCO_ROOT}: {missing_annotation_files}"
    )

with instances_train_path.open("r", encoding="utf-8") as file:
    instances = json.load(file)
with captions_train_path.open("r", encoding="utf-8") as file:
    captions_data = json.load(file)
with instances_val_path.open("r", encoding="utf-8") as file:
    instances_val = json.load(file)
with captions_val_path.open("r", encoding="utf-8") as file:
    captions_val_data = json.load(file)

def captions_by_id(caption_dataset):
    result = defaultdict(list)
    for annotation in caption_dataset["annotations"]:
        result[annotation["image_id"]].append(annotation["caption"])
    return result

train_file_to_id = {image["file_name"]: image["id"] for image in captions_data["images"]}
val_file_to_id = {image["file_name"]: image["id"] for image in captions_val_data["images"]}
train_captions_by_id = captions_by_id(captions_data)
val_captions_by_id = captions_by_id(captions_val_data)

def make_reference_records(paths, file_to_id, captions_for_id, split_name):
    records = []
    caption_count_issues = []
    for image_path in paths:
        image_id = file_to_id.get(image_path.name)
        if image_id is None:
            raise ValueError(f"No COCO {split_name} image ID for {image_path.name}")
        raw_captions = captions_for_id.get(image_id, [])
        if len(raw_captions) < 5:
            caption_count_issues.append(image_path.name)
        records.append(
            {
                "image_id": image_id,
                "file_name": image_path.name,
                "captions": raw_captions[:5],
                "source": f"captions_{split_name}.json",
                "confidence": CAPTION_REFERENCE_CONFIDENCE,
            }
        )
    return records, caption_count_issues

kb_reference_records, kb_caption_issues = make_reference_records(
    kb_images, train_file_to_id, train_captions_by_id, "train2017"
 )
validation_reference_records, validation_caption_issues = make_reference_records(
    val_images, train_file_to_id, train_captions_by_id, "train2017"
 )
test_reference_records, test_caption_issues = make_reference_records(
    test_1000, val_file_to_id, val_captions_by_id, "val2017"
 )
if kb_caption_issues or validation_caption_issues or test_caption_issues:
    raise ValueError(
        "Some selected images have fewer than five reference captions: "
        f"KB={len(kb_caption_issues)}, validation={len(validation_caption_issues)}, "
        f"test={len(test_caption_issues)}"
    )

kb_ids = {record["image_id"] for record in kb_reference_records}
validation_ids = {record["image_id"] for record in validation_reference_records}
test_ids = {record["image_id"] for record in test_reference_records}
assert not (kb_ids & validation_ids)
assert not (kb_ids & test_ids)
assert not (validation_ids & test_ids)

for filename, records in (
    ("kb_captions.json", kb_reference_records),
    ("validation_references.json", validation_reference_records),
    ("test_references_1000.json", test_reference_records),
    ("test_references_100_compare.json", test_reference_records[:TARGET_TEST_100]),
):
    with (OUTPUT_DIR / filename).open("w", encoding="utf-8") as file:
        json.dump(records, file, ensure_ascii=False, indent=2)

print(f"Caption reference confidence: {CAPTION_REFERENCE_CONFIDENCE:.1f} (human-written COCO ground truth).")
print(f"KB caption records: {len(kb_reference_records):,}; exactly five captions per image")
print(f"Validation references: {len(validation_reference_records):,}")
print(f"Test references saved separately: {len(test_reference_records):,}")

category_names = {category["id"]: category["name"] for category in instances["categories"]}
object_count_by_id = Counter(annotation["image_id"] for annotation in instances["annotations"])
category_counts = Counter(
    category_names.get(annotation["category_id"], "unknown")
    for annotation in instances["annotations"]
 )
caption_lengths_by_id = defaultdict(list)
all_captions = [annotation["caption"] for annotation in captions_data["annotations"]]
all_word_counts = [len(caption.split()) for caption in all_captions]
for annotation in captions_data["annotations"]:
    caption_lengths_by_id[annotation["image_id"]].append(len(annotation["caption"].split()))

train_image_names = {image["file_name"] for image in instances["images"]}
train_disk_names = {path.name for path in TRAIN_IMG_DIR.glob("*.jpg")}
val_image_names = {image["file_name"] for image in instances_val["images"]}
val_disk_names = {path.name for path in VAL_IMG_DIR.glob("*.jpg")}
missing_train_files = sorted(train_image_names - train_disk_names)
missing_val_files = sorted(val_image_names - val_disk_names)

scene_rows = []
for image in instances["images"]:
    image_id = image["id"]
    lengths = caption_lengths_by_id.get(image_id, [])
    scene_rows.append(
        {
            "image_id": image_id,
            "file_name": image["file_name"],
            "object_count": object_count_by_id.get(image_id, 0),
            "caption_count": len(lengths),
            "mean_caption_words": float(np.mean(lengths)) if lengths else 0.0,
        }
    )

vocabulary = {
    token.strip(".,!?;:\"'()[]{}").lower()
    for caption in all_captions
    for token in caption.split()
    if token.strip(".,!?;:\"'()[]{}")
}
top_50_categories = category_counts.most_common(50)
object_counts = [row["object_count"] for row in scene_rows]
statistics = {
    "train_images_in_annotations": len(instances["images"]),
    "train_images_on_disk": len(train_disk_names),
    "missing_train_image_files": len(missing_train_files),
    "val_images_in_annotations": len(instances_val["images"]),
    "val_images_on_disk": len(val_disk_names),
    "missing_val_image_files": len(missing_val_files),
    "train_caption_count": len(all_captions),
    "train_caption_reference_count_by_image": dict(Counter(
        len(value) for value in train_captions_by_id.values()
    )),
    "average_caption_words": float(np.mean(all_word_counts)),
    "vocabulary_size_whitespace_tokens": len(vocabulary),
    "average_objects_per_train_image": float(np.mean(object_counts)),
    "median_objects_per_train_image": float(np.median(object_counts)),
    "top_50_object_categories": [
        {"category": name, "count": count} for name, count in top_50_categories
    ],
    "kb_images_with_more_than_five_captions": sum(
        len(train_captions_by_id[record["image_id"]]) > 5 for record in kb_reference_records
    ),
}
with (OUTPUT_DIR / "preprocessing_statistics.json").open("w", encoding="utf-8") as file:
    json.dump(statistics, file, indent=2)

with (OUTPUT_DIR / "train_image_statistics.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=scene_rows[0].keys())
    writer.writeheader()
    writer.writerows(scene_rows)

with (OUTPUT_DIR / "train_missing_image_files.json").open("w", encoding="utf-8") as file:
    json.dump({"train2017": missing_train_files, "val2017": missing_val_files}, file, indent=2)

cleaning_audit.update(
    {
        "annotation_image_integrity": {
            "missing_train_files": len(missing_train_files),
            "missing_val_files": len(missing_val_files),
            "kb_images_missing_captions": len(kb_caption_issues),
            "validation_images_missing_captions": len(validation_caption_issues),
            "test_images_missing_captions": len(test_caption_issues),
            "test_reference_captions_saved_outside_kb": True,
        }
    }
 )
with (OUTPUT_DIR / "cleaning_audit.json").open("w", encoding="utf-8") as file:
    json.dump(cleaning_audit, file, indent=2)

plt.figure(figsize=(8, 4))
sns.histplot(all_word_counts, bins=30, kde=True, color="steelblue")
plt.title("Before preprocessing: COCO caption lengths")
plt.xlabel("Words per caption")
plt.ylabel("Caption count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "before_prep_caption_lengths.png", dpi=300)
plt.show()

names, counts = zip(*top_50_categories)
plt.figure(figsize=(11, 12))
sns.barplot(x=list(counts), y=list(names), color="darkorange")
plt.title("Before preprocessing: top 50 COCO object categories")
plt.xlabel("Annotation count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "before_prep_objects_top50.png", dpi=300)
plt.show()

plt.figure(figsize=(8, 4))
sns.histplot(object_counts, bins=40, kde=False, color="slateblue")
plt.title("Before preprocessing: objects per train image")
plt.xlabel("COCO object annotations per image")
plt.ylabel("Image count")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "before_prep_scene_complexity.png", dpi=300)
plt.show()

sample_size = min(20_000, len(scene_rows))
sample_indices = rng.choice(len(scene_rows), size=sample_size, replace=False)
plt.figure(figsize=(8, 5))
plt.scatter(
    [scene_rows[index]["object_count"] for index in sample_indices],
    [scene_rows[index]["mean_caption_words"] for index in sample_indices],
    alpha=0.25,
    s=10,
    color="teal",
 )
plt.title("Before preprocessing: scene complexity vs. caption length")
plt.xlabel("COCO object annotations per image")
plt.ylabel("Average caption length (words)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "before_prep_scene_vs_caption.png", dpi=300)
plt.show()

print(f"Mean caption length: {statistics['average_caption_words']:.2f} words")
print(f"Whitespace-token vocabulary size: {statistics['vocabulary_size_whitespace_tokens']:,}")
print(f"Mean scene complexity: {statistics['average_objects_per_train_image']:.2f} objects/image")
print(f"Top 50 categories and per-image statistics saved under {OUTPUT_DIR.resolve()}.")

In [ ]:
print("Extracting objects from all knowledge-base images with YOLOv8n...")
model = YOLO("yolov8n.pt")
DETECTION_BATCH_SIZE = 128
sample_kb = kb_images[:YOLO_SAMPLE_SIZE]
metadata_path = OUTPUT_DIR / "extracted_metadata_yolo.json"
if metadata_path.exists():
    with metadata_path.open("r", encoding="utf-8") as file:
        extracted_metadata = json.load(file)
else:
    extracted_metadata = {}

sample_names = {image_path.name for image_path in sample_kb}
if not set(extracted_metadata).issubset(sample_names):
    raise ValueError("Existing YOLO metadata does not match the frozen knowledge-base split.")
for record in extracted_metadata.values():
    if (
        record.get("extraction_model") != "YOLOv8n"
        or record.get("confidence_threshold") != DETECTION_CONFIDENCE
    ):
        raise ValueError("Existing YOLO metadata was produced with a different detector configuration.")

detected_objects = [
    obj
    for record in extracted_metadata.values()
    for obj in record["detected_objects"]
]
remaining_kb = [image_path for image_path in sample_kb if image_path.name not in extracted_metadata]
print(f"Resuming with {len(extracted_metadata):,} existing detections; {len(remaining_kb):,} images remain.")

for batch_start in range(0, len(remaining_kb), DETECTION_BATCH_SIZE):
    batch_paths = remaining_kb[batch_start:batch_start + DETECTION_BATCH_SIZE]
    detection_stream = model.predict(
        source=[str(image_path) for image_path in batch_paths],
        stream=True,
        batch=DETECTION_BATCH_SIZE,
        conf=DETECTION_CONFIDENCE,
        verbose=False,
    )
    for image_path, result in zip(batch_paths, detection_stream, strict=True):
        class_names = result.names
        if result.boxes is None:
            image_objects = []
            detection_confidences = []
        else:
            class_ids = result.boxes.cls.tolist()
            detection_confidences = result.boxes.conf.tolist()
            image_objects = [class_names[int(class_id)] for class_id in class_ids]
        extracted_metadata[image_path.name] = {
            "image_id": train_file_to_id[image_path.name],
            "detected_objects": image_objects,
            "detection_confidences": detection_confidences,
            "extraction_model": "YOLOv8n",
            "confidence_threshold": DETECTION_CONFIDENCE,
        }
        detected_objects.extend(image_objects)

    with metadata_path.open("w", encoding="utf-8") as file:
        json.dump(extracted_metadata, file)
    processed = len(extracted_metadata)
    print(f"Processed {processed:,}/{len(sample_kb):,} knowledge-base images")

print(f"Saved detections for {len(extracted_metadata):,} knowledge-base images.")
print(f"Detector confidence threshold: {DETECTION_CONFIDENCE}")

In [ ]:
print("Exploring the objects detected by YOLO...")
yolo_counts = Counter(detected_objects)
top_yolo = yolo_counts.most_common(15)

if top_yolo:
    names, counts = zip(*top_yolo)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=list(counts), y=list(names), color="seagreen")
    plt.title("After preprocessing: top YOLO object detections")
    plt.xlabel("Detection count")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "after_prep_extracted_objects.png", dpi=300)
    plt.show()
else:
    print("No objects were detected in the selected sample.")

print(f"Charts and data are saved under {OUTPUT_DIR.resolve()}.")

In [ ]:
import time

import faiss
from sentence_transformers import SentenceTransformer

print("Generating embeddings and building the FAISS index...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
kb_texts = []
for image_path in sample_kb:
    objects = extracted_metadata[image_path.name]["detected_objects"]
    object_text = ", ".join(objects) if objects else "no objects detected"
    kb_texts.append(f"Image {image_path.name} contains objects: {object_text}")

start_time = time.time()
embeddings = embedder.encode(
    kb_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True,
 )
indexing_time = time.time() - start_time

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype("float32"))
faiss.write_index(index, str(OUTPUT_DIR / "faiss_kb.index"))
np.save(OUTPUT_DIR / "kb_embeddings.npy", embeddings)

print(f"FAISS index built in {indexing_time:.2f} seconds.")
print(f"Vectors indexed: {index.ntotal:,}")
print(f"Embedding dimension: {embeddings.shape[1]}")
print(f"Index and embeddings saved under {OUTPUT_DIR.resolve()}.")